# SmolVLA Model/Data Diagnosis: A -> E

Notebook này chạy đủ 5 section trước khi làm VLA Harness runtime.

**Lưu ý commit HF:** mỗi lần upload model có 3 commit. Phải dùng commit cuối mỗi cụm để có đủ weights/config + preprocessor + postprocessor.

- Old final: `b6f2aafdbdd793046747fad8207459402c33c4b0`
- New final: `f7029d03d69e149cb4b7cea8747d7158d35a8fd0`

## 0. Setup

Nếu không chạy ở `/home/trietlm/lerobot`, set env `LEROBOT_ROOT` hoặc sửa `cfg.repo_root`. Nếu dataset không ở path mặc định, set env `LEROBOT_DATASET_ROOT`.

In [1]:
from pathlib import Path
import sys
from IPython.display import Markdown, display

REPO_ROOT = Path('/home/trietlm/lerobot')
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'xai'))

from smolvla_model_data_diagnosis import (
    DiagnosisConfig, OLD_FINAL_REV, NEW_FINAL_REV, choose_probe_rows,
    run_replay, run_state_scan, run_dataset_audit, run_config_audit, write_eval_report,
)

cfg = DiagnosisConfig(repo_root=REPO_ROOT)
print('repo_root:', cfg.repo_root)
print('output_dir:', cfg.output_dir)
print('device:', cfg.device)
print('run_dirs:', [p.name for p in cfg.run_dirs])
print('dataset_root:', cfg.dataset_root, 'exists=', cfg.dataset_root.exists())
print('safe_pose:', cfg.safe_pose_path, 'exists=', cfg.safe_pose_path.exists())
print('policy specs:', cfg.policy_specs)
for run_dir in cfg.run_dirs:
    probes = choose_probe_rows(run_dir)
    print(run_dir.name, 'n_probes=', len(probes), 'first_timesteps=', [p['timestep'] for p in probes[:20]])

/home/trietlm/lerobot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo_root: /home/trietlm/lerobot
output_dir: /home/trietlm/lerobot/xai/model_data_diagnosis_smolvla_0629_outputs
device: cuda
run_dirs: ['recorded_obs-0629-01', 'recorded_obs-0629-02', 'recorded_obs-0629-nonrtc', 'recorded_obs-0629-rtc2']
dataset_root: /home/trietlm/lerobot/data/so-arm-101-pouring-0.3-cutted exists= True
safe_pose: /home/trietlm/lerobot/safe_pose.json exists= True
policy specs: [{'label': 'old_50eps_final', 'repo_id': 'di-techinnova/smolvla-pouring-0.3-cutted', 'revision': 'b6f2aafdbdd793046747fad8207459402c33c4b0'}, {'label': 'new_200eps_final', 'repo_id': 'di-techinnova/smolvla-pouring-0.3-cutted', 'revision': 'f7029d03d69e149cb4b7cea8747d7158d35a8fd0'}]
recorded_obs-0629-01 n_probes= 40 first_timesteps= [0, 13, 17, 32, 34, 49, 51, 66, 68, 83, 85, 100, 102, 117, 119, 134, 136, 151, 153, 168]
recorded_obs-0629-02 n_probes= 27 first_timesteps= [0, 16, 18, 33, 35, 50, 52, 67, 69, 84, 86, 101, 103, 118, 120, 135, 137, 152, 154, 169]
recorded_obs-0629-nonrtc n_probes= 21 

## A. Old vs New Replay

Replay old/new trên cùng recorded obs. Section này xuất `A_replay_predictions.csv`, `A_replay_summary.csv`, `A_server_compare.csv`, và plots vào `plots_A_replay/`.

In [2]:
replay_df, replay_summary, run_policy_agg, server_compare = run_replay(cfg)
print('replay rows:', len(replay_df), 'errors:', int(replay_df['error'].notna().sum()))
display(run_policy_agg)
if not server_compare.empty:
    display(
        server_compare.groupby(['run', 'server_rtc_enabled', 'policy', 'image_variant'], dropna=False)
        .agg(
            n=('obs_timestep', 'count'),
            abs_delta_end_mean=('delta_end_server_minus_offline', lambda x: float(abs(x).mean())),
            delta_end_mean=('delta_end_server_minus_offline', 'mean'),
            server_safe_pull_count=('server_safe_pull', 'sum'),
            offline_safe_pull_count=('offline_safe_pull', 'sum'),
        )
        .reset_index()
    )
    display(server_compare.sort_values('delta_end_server_minus_offline', key=lambda s: s.abs(), ascending=False).head(40))

\n=== Loading policy: old_50eps_final b6f2aafdbdd793046747fad8207459402c33c4b0 ===


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 38756.40it/s]


Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
Loading weights: 100%|██████████| 489/489 [00:00<00:00, 4761.25it/s]


Reducing the number of VLM layers to 16 ...
Loading weights from local directory
old_50eps_final recorded_obs-0629-01 n_probes 40
old_50eps_final recorded_obs-0629-02 n_probes 27
old_50eps_final recorded_obs-0629-nonrtc n_probes 21
old_50eps_final recorded_obs-0629-rtc2 n_probes 18
\n=== Loading policy: new_200eps_final f7029d03d69e149cb4b7cea8747d7158d35a8fd0 ===


...

Loading weights: 100%|██████████| 489/489 [00:00<00:00, 4096.18it/s]


Reducing the number of VLM layers to 16 ...
Loading weights from local directory
new_200eps_final recorded_obs-0629-01 n_probes 40
new_200eps_final recorded_obs-0629-02 n_probes 27
new_200eps_final recorded_obs-0629-nonrtc n_probes 21
new_200eps_final recorded_obs-0629-rtc2 n_probes 18
replay rows: 16960 errors: 0


,run,policy,n_obs,safe_pull_count,safe_pull_rate,mean_end_minus_current,max_end_minus_current,mean_chunk_step
0,recorded_obs-0629-01,new_200eps_final,40,22,0.550000,0.112793,0.687050,4.153959
1,recorded_obs-0629-01,old_50eps_final,40,0,0.000000,-0.232267,-0.004598,3.834434
2,recorded_obs-0629-02,new_200eps_final,27,6,0.222222,-0.081623,0.566063,3.675039
3,recorded_obs-0629-02,old_50eps_final,27,0,0.000000,-0.226337,0.011036,4.349495
4,recorded_obs-0629-nonrtc,new_200eps_final,21,7,0.333333,0.029830,0.636897,3.629282
5,recorded_obs-0629-nonrtc,old_50eps_final,21,0,0.000000,-0.190630,0.036197,4.606528
6,recorded_obs-0629-rtc2,new_200eps_final,18,7,0.388889,0.050521,0.409974,3.776915
7,recorded_obs-0629-rtc2,old_50eps_final,18,0,0.000000,-0.209325,0.043936,4.819645


,run,server_rtc_enabled,policy,image_variant,n,abs_delta_end_mean,delta_end_mean,server_safe_pull_count,offline_safe_pull_count
0,recorded_obs-0629-01,True,new_200eps_final,saved_rgb,40,0.159532,-0.081665,19,22
1,recorded_obs-0629-01,True,new_200eps_final,server_jpeg_bgr_q90,40,0.116622,0.003617,19,13
2,recorded_obs-0629-01,True,old_50eps_final,saved_rgb,40,0.297429,0.263395,19,0
3,recorded_obs-0629-01,True,old_50eps_final,server_jpeg_bgr_q90,40,0.296917,0.255340,19,0
4,recorded_obs-0629-02,True,new_200eps_final,saved_rgb,27,0.095244,-0.017167,5,6
5,recorded_obs-0629-02,True,new_200eps_final,server_jpeg_bgr_q90,27,0.083523,-0.001645,5,4
6,recorded_obs-0629-02,True,old_50eps_final,saved_rgb,27,0.185341,0.127547,5,0
7,recorded_obs-0629-02,True,old_50eps_final,server_jpeg_bgr_q90,27,0.179402,0.096038,5,0
8,recorded_obs-0629-nonrtc,False,new_200eps_final,saved_rgb,21,0.088037,0.033233,9,7
9,recorded_obs-0629-nonrtc,False,new_200eps_final,server_jpeg_bgr_q90,21,0.095748,0.049380,9,6


,run,obs_timestep,server_rtc_enabled,server_rtc_real_delay,current_p,server_end_p,server_end_minus_current_p,policy,image_variant,offline_end_mean,offline_end_minus_current_p,delta_end_server_minus_offline,server_safe_pull,offline_safe_pull
187,recorded_obs-0629-02,52,True,3.0,0.299960,0.950940,0.650980,old_50eps_final,server_jpeg_bgr_q90,-0.010856,-0.310816,0.961795,True,False
186,recorded_obs-0629-02,52,True,3.0,0.299960,0.950940,0.650980,old_50eps_final,saved_rgb,-0.004084,-0.304044,0.955024,True,False
374,recorded_obs-0629-rtc2,34,True,3.0,0.384905,0.922737,0.537832,old_50eps_final,saved_rgb,-0.028279,-0.413184,0.951016,True,False
375,recorded_obs-0629-rtc2,34,True,3.0,0.384905,0.922737,0.537832,old_50eps_final,server_jpeg_bgr_q90,-0.024094,-0.408999,0.946831,True,False
362,recorded_obs-0629-rtc2,15,True,3.0,0.291576,0.829314,0.537738,old_50eps_final,saved_rgb,-0.076016,-0.367592,0.905329,True,False
363,recorded_obs-0629-rtc2,15,True,3.0,0.291576,0.829314,0.537738,old_50eps_final,server_jpeg_bgr_q90,-0.073810,-0.365386,0.903124,True,False
63,recorded_obs-0629-01,134,True,3.0,0.444309,0.934065,0.489757,old_50eps_final,server_jpeg_bgr_q90,0.032640,-0.411669,0.901426,True,False
62,recorded_obs-0629-01,134,True,3.0,0.444309,0.934065,0.489757,old_50eps_final,saved_rgb,0.037055,-0.407253,0.897010,True,False
370,recorded_obs-0629-rtc2,32,True,3.0,0.402900,0.865716,0.462816,old_50eps_final,saved_rgb,-0.027543,-0.430442,0.893259,True,False
287,recorded_obs-0629-nonrtc,38,False,NaN,0.223850,0.828981,0.605131,old_50eps_final,server_jpeg_bgr_q90,-0.062970,-0.286820,0.891951,True,False


## B. State Scan

Giữ ảnh cố định, quét state từ `start_pose -> safe_pose` để tìm vùng state khiến model kéo safe.

In [3]:
state_scan_df, state_scan_summary, danger = run_state_scan(cfg)
print('state scan rows:', len(state_scan_df), 'errors:', int(state_scan_df['error'].notna().sum()))
display(danger)
display(state_scan_summary.head())

\n=== State scan policy: old_50eps_final ===


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 41943.04it/s]


Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


Loading weights: 100%|██████████| 489/489 [00:00<00:00, 4281.96it/s]


Reducing the number of VLM layers to 16 ...
Loading weights from local directory
old_50eps_final recorded_obs-0629-nonrtc image timesteps [0, 17, 36, 55]
old_50eps_final recorded_obs-0629-rtc2 image timesteps [0, 15, 32, 49]
\n=== State scan policy: new_200eps_final ===


...

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


Loading weights: 100%|██████████| 489/489 [00:00<00:00, 4243.02it/s]


Reducing the number of VLM layers to 16 ...
Loading weights from local directory
new_200eps_final recorded_obs-0629-nonrtc image timesteps [0, 17, 36, 55]
new_200eps_final recorded_obs-0629-rtc2 image timesteps [0, 15, 32, 49]
state scan rows: 3840 errors: 0


,run,policy,image_timestep,first_danger_alpha,max_delta
0,recorded_obs-0629-nonrtc,new_200eps_final,0,-0.050000,0.298769
1,recorded_obs-0629-nonrtc,new_200eps_final,17,-0.050000,0.342774
2,recorded_obs-0629-nonrtc,new_200eps_final,36,-0.050000,0.254630
3,recorded_obs-0629-nonrtc,new_200eps_final,55,-0.050000,0.172194
4,recorded_obs-0629-nonrtc,old_50eps_final,0,0.659574,0.065069
5,recorded_obs-0629-rtc2,new_200eps_final,0,-0.050000,0.546392
6,recorded_obs-0629-rtc2,new_200eps_final,15,-0.050000,0.179581
7,recorded_obs-0629-rtc2,new_200eps_final,32,-0.050000,0.266295
8,recorded_obs-0629-rtc2,new_200eps_final,49,-0.050000,0.085383


,run,policy,policy_revision,image_timestep,image_elapsed_s,alpha_current_p,p_first_mean,p_end_mean,p_max_mean,end_minus_current_p,safe_pull_rate,chunk_step_mean,n,safe_pull
0,recorded_obs-0629-nonrtc,new_200eps_final,f7029d03d69e149cb4b7cea8747d7158d35a8fd0,0,0.0,-0.050000,-0.039699,0.248769,0.294860,0.298769,1.0,4.052653,5,True
1,recorded_obs-0629-nonrtc,new_200eps_final,f7029d03d69e149cb4b7cea8747d7158d35a8fd0,0,0.0,-0.025532,-0.019447,0.184178,0.277676,0.209710,1.0,4.128618,5,True
2,recorded_obs-0629-nonrtc,new_200eps_final,f7029d03d69e149cb4b7cea8747d7158d35a8fd0,0,0.0,-0.001064,0.003371,0.153446,0.279783,0.154509,1.0,4.236282,5,True
3,recorded_obs-0629-nonrtc,new_200eps_final,f7029d03d69e149cb4b7cea8747d7158d35a8fd0,0,0.0,0.023404,0.025777,0.135613,0.282573,0.112209,1.0,4.272966,5,True
4,recorded_obs-0629-nonrtc,new_200eps_final,f7029d03d69e149cb4b7cea8747d7158d35a8fd0,0,0.0,0.047872,0.047543,0.139024,0.296571,0.091151,0.8,4.385787,5,True


## C. Dataset Bucket Audit

Đọc train parquet, project state/action theo trục `episode_start -> safe_pose`, rồi tìm bucket/episode có action kéo về safe/rest. Nếu dataset root thiếu, sửa `cfg.dataset_root` hoặc set `cfg.download_dataset_if_missing=True`.

In [4]:
train_df, dataset_audit, bucket_summary, episode_suspects = run_dataset_audit(cfg)
print('train rows:', len(train_df), 'audit rows:', len(dataset_audit))
if not bucket_summary.empty:
    display(bucket_summary)
if not episode_suspects.empty:
    display(episode_suspects.head(40))

train rows: 73803 audit rows: 73803


,p_bucket,n,safe_pull_count,safe_pull_rate,mean_delta_p,max_delta_p,mean_action_dist_safe
0,"(-0.201, -0.15]",4283,258,0.060238,-0.000542,0.287489,197.200012
1,"(-0.15, -0.1]",3587,202,0.056314,0.000063,0.347780,190.730653
2,"(-0.1, -0.05]",3697,137,0.037057,-0.000038,0.329266,183.912211
3,"(-0.05, -2.78e-17]",3362,114,0.033908,-0.001898,0.322193,171.428214
4,"(-2.78e-17, 0.05]",5204,274,0.052652,0.003487,0.682833,155.243956
5,"(0.05, 0.1]",2808,257,0.091524,0.001537,0.384600,155.538208
6,"(0.1, 0.15]",2143,202,0.094260,-0.000307,0.219283,148.907954
7,"(0.15, 0.2]",1949,179,0.091842,-0.001707,0.477380,141.050856
8,"(0.2, 0.25]",1419,147,0.103594,-0.002218,0.178676,132.904517
9,"(0.25, 0.3]",1142,103,0.090193,-0.003533,0.409681,124.190556


,episode_index,n,safe_pull_count,safe_pull_rate,max_delta_p,min_p_state,max_p_state
51,51,441,236,0.535147,1.017421,-12.001848,0.467059
0,0,439,236,0.537585,0.929942,-10.326026,0.020096
57,57,402,190,0.472637,0.515752,-7.126908,0.019032
90,90,406,177,0.435961,0.794941,-13.373161,0.000000
145,145,434,113,0.260369,0.592074,-9.767965,0.000000
60,60,422,99,0.234597,0.731762,-12.169215,0.000000
120,120,400,91,0.227500,0.458554,-9.076486,0.000000
59,59,380,88,0.231579,0.822923,-9.173598,0.000000
50,50,428,83,0.193925,0.682833,-5.121741,0.417227
58,58,382,76,0.198953,0.641869,-8.578678,0.000000


## D. Config / Processor Audit

So old/new final revisions về files, config, preprocessor/postprocessor. Đây là nơi kiểm tra có bị load nhầm commit giữa cụm 3 commit không.

In [5]:
snapshot_df, file_manifest, config_flat, config_diff, file_diff = run_config_audit(cfg)
display(snapshot_df)
print('tracked files:', len(file_manifest), 'changed/missing files:', len(file_diff), 'changed config keys:', len(config_diff))
display(file_diff.head(100))
display(config_diff.head(200))

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 21657.34it/s]


,policy,revision,snapshot_path,n_files
0,old_50eps_final,b6f2aafdbdd793046747fad8207459402c33c4b0,/home/trietlm/.cache/huggingface/hub/models--d...,9
1,new_200eps_final,f7029d03d69e149cb4b7cea8747d7158d35a8fd0,/home/trietlm/.cache/huggingface/hub/models--d...,9


tracked files: 14 changed/missing files: 5 changed config keys: 14


policy,relpath,new_200eps_final,old_50eps_final,changed_or_missing
0,README.md,d40e408ed29338ea8994005e7ff94564584ea104b20b67...,caf10e96227eac3a691b66a772c6bca266211d9ca82f30...,True
1,config.json,836435d616b288a015c1fb760be0bc924c042b94cd4592...,0af0e9fa28026ee41874808485f11b8776075456bb94c9...,True
3,policy_postprocessor_step_0_unnormalizer_proce...,92ad922f0c4e9b2520ccf7ff6f4a2677544a4ea244f91a...,277d404581037707298a0e0941ef9516e6cc8353bd1664...,True
5,policy_preprocessor_step_5_normalizer_processo...,ba4c837b99e82db2bf53c912777824fa8e1a8c673bdc18...,3820f04aa3b43b82956661855131d3eb416ee0af621334...,True
6,train_config.json,0cb58e6a642356124b45972674f77e0c7732db9e6b5318...,6ee2ba2b0972616a500f78aac0b6ed6c00ab507b20ee21...,True


policy,file,key,new_200eps_final,old_50eps_final,changed
3,config.json,chunk_size,35,30,True
24,config.json,n_action_steps,35,30,True
48,config.json,scheduler_decay_steps,95000,30000,True
49,config.json,scheduler_warmup_steps,5000,1000,True
143,train_config.json,job_name,"""smolvla-pouring-0.3-cutted-v2""","""smolvla-pouring-0.3-cutted""",True
152,train_config.json,output_dir,"""outputs/train/smolvla-pouring-0.3-cutted-v2""","""outputs/train/smolvla-pouring-0.3-cutted""",True
158,train_config.json,policy.chunk_size,35,30,True
179,train_config.json,policy.n_action_steps,35,30,True
203,train_config.json,policy.scheduler_decay_steps,95000,30000,True
204,train_config.json,policy.scheduler_warmup_steps,5000,1000,True


## E. Offline Eval Report

Gom các section thành report markdown + policy score.

In [6]:
report_text = write_eval_report(
    cfg,
    run_policy_agg=run_policy_agg,
    danger=danger,
    suspects=episode_suspects,
    config_diff=config_diff,
    file_diff=file_diff,
)
print('wrote:', cfg.output_dir / 'E_offline_eval_report.md')
display(Markdown(report_text))

ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.